In [ ]:
from google.colab import drive
drive.mount('/content/drive')
work_dir = "/content/drive/MyDrive/npm_lsh_work"

Mounted at /content/drive


In [ ]:
import pandas as pd

pypi_results = pd.read_parquet('part-00000-tid-4486452325674416850-2b7efa37-6552-4f06-a133-06d532c30c8a-460-1.c000.snappy.parquet')
npm_results = pd.read_parquet(f'{work_dir}/npm_lsh_matches.parquet')

print("PyPI rows:", len(pypi_results))
print("npm rows:", len(npm_results))
print(pypi_results.head())

PyPI rows: 471176
npm rows: 8537
        popular_name       candidate_name  jaccard_distance  length_ratio
0  clarifai-protocol         0b1-protocol          0.578947      1.416667
1  langchain-litellm  0din-litellm-shield          0.625000      1.117647
2             opencc               0pencv          0.571429      1.000000
3    etcd-sdk-python           0x0-python          0.647059      1.500000
4            bpython           0x0-python          0.500000      1.428571


In [ ]:
pypi_shortlist = pypi_results.nsmallest(75, "jaccard_distance")[["popular_name", "candidate_name", "jaccard_distance"]].copy()
pypi_shortlist["ecosystem"] = "pypi"

npm_shortlist = npm_results.nsmallest(75, "jaccard_distance")[["popular_name", "candidate_name", "jaccard_distance"]].copy()
npm_shortlist["ecosystem"] = "npm"

combined_shortlist = pd.concat([pypi_shortlist, npm_shortlist], ignore_index=True)
combined_shortlist.to_csv(f"{work_dir}/signal3_shortlist.csv", index=False)

print("Combined shortlist size:", len(combined_shortlist))
print(combined_shortlist.head(10))

Combined shortlist size: 150
                           popular_name                 candidate_name  \
0  apache-airflow-providers-apache-hdfs  apache-airflow-providers-hdfs   
1                                  comm                          commm   
2                               pexpect                         expect   
3                                  lxml                          lxmlx   
4                                 manim                          anima   
5                            etelemetry                      telemetry   
6                                 wheel                         wheeel   
7                                 babel                        bababel   
8                                 manim                         imanim   
9                     nvidia-cufft-cu11             nvidia-cufft-cu111   

   jaccard_distance ecosystem  
0               0.0      pypi  
1               0.0      pypi  
2               0.0      pypi  
3               0.0      pyp

In [ ]:
print(len(GITHUB_TOKEN))
print(repr(GITHUB_TOKEN[:10]), repr(GITHUB_TOKEN[-4:]))

21
'paste_your' 'here'


In [ ]:
GITHUB_TOKEN = "REDACTED"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

import requests
test_response = requests.get("https://api.github.com/rate_limit", headers=headers)
print(test_response.status_code)
print(test_response.json())

200
{'resources': {'core': {'limit': 5000, 'used': 0, 'remaining': 5000, 'reset': 1789321842}, 'search': {'limit': 30, 'used': 0, 'remaining': 30, 'reset': 1789318302}, 'graphql': {'limit': 5000, 'used': 0, 'remaining': 5000, 'reset': 1789321842}, 'integration_manifest': {'limit': 5000, 'used': 0, 'remaining': 5000, 'reset': 1789321842}, 'source_import': {'limit': 100, 'used': 0, 'remaining': 100, 'reset': 1789318302}, 'code_scanning_autofix': {'limit': 10, 'used': 0, 'remaining': 10, 'reset': 1789318302}, 'actions_runner_registration': {'limit': 10000, 'used': 0, 'remaining': 10000, 'reset': 1789321842}, 'scim': {'limit': 15000, 'used': 0, 'remaining': 15000, 'reset': 1789321842}, 'dependency_snapshots': {'limit': 100, 'used': 0, 'remaining': 100, 'reset': 1789318302}, 'dependency_sbom': {'limit': 100, 'used': 0, 'remaining': 100, 'reset': 1789318302}, 'audit_log': {'limit': 1750, 'used': 0, 'remaining': 1750, 'reset': 1789321842}, 'audit_log_streaming': {'limit': 15, 'used': 0, 'rema

In [ ]:
import time

results = []

for idx, row in combined_shortlist.iterrows():
    candidate = row["candidate_name"]
    ecosystem = row["ecosystem"]
    filename = "requirements.txt" if ecosystem == "pypi" else "package.json"

    query = f'"{candidate}" in:file filename:{filename}'
    response = requests.get(
        "https://api.github.com/search/code",
        headers=headers,
        params={"q": query}
    )

    if response.status_code == 200:
        data = response.json()
        hit_count = data.get("total_count", 0)
        example_repo = data["items"][0]["repository"]["full_name"] if hit_count > 0 else None
    else:
        hit_count = None
        example_repo = f"ERROR {response.status_code}"

    results.append({
        "popular_name": row["popular_name"],
        "candidate_name": candidate,
        "ecosystem": ecosystem,
        "jaccard_distance": row["jaccard_distance"],
        "github_hit_count": hit_count,
        "example_repo": example_repo
    })

    if (idx + 1) % 10 == 0:
        print(f"Checked {idx + 1}/150...")

    time.sleep(2.5)  # stay safely under the 30/minute search rate limit

evidence_df = pd.DataFrame(results)
evidence_df.to_csv(f"{work_dir}/signal3_evidence_results.csv", index=False)
print("Done — all 150 candidates checked")

Checked 10/150...
Checked 20/150...
Checked 30/150...
Checked 40/150...
Checked 50/150...
Checked 60/150...
Checked 70/150...
Checked 80/150...
Checked 90/150...
Checked 100/150...
Checked 110/150...
Checked 120/150...
Checked 130/150...
Checked 140/150...
Checked 150/150...
Done — all 150 candidates checked


In [ ]:
print("Total checked:", len(evidence_df))
print("Candidates with real GitHub evidence (hit_count > 0):", (evidence_df["github_hit_count"] > 0).sum())
print("Candidates with zero hits:", (evidence_df["github_hit_count"] == 0).sum())
print("Candidates with errors:", evidence_df["github_hit_count"].isna().sum())

print("\nTop candidates WITH real evidence:")
print(evidence_df[evidence_df["github_hit_count"] > 0].sort_values("github_hit_count", ascending=False).head(15))

Total checked: 150
Candidates with real GitHub evidence (hit_count > 0): 53
Candidates with zero hits: 26
Candidates with errors: 71

Top candidates WITH real evidence:
          popular_name    candidate_name ecosystem  jaccard_distance  \
2              pexpect            expect      pypi          0.000000   
5           etelemetry         telemetry      pypi          0.000000   
46      elasticsearch9     elasticsearch      pypi          0.076923   
61        nvidia-ml-py     nvidia-ml-py3      pypi          0.083333   
24   django-bootstrap3  django-bootstrap      pypi          0.062500   
25   django-bootstrap5  django-bootstrap      pypi          0.062500   
4                manim             anima      pypi          0.000000   
45      django-tables2     django-tables      pypi          0.076923   
58       progressbar33      progressbar3      pypi          0.083333   
64       flake8-commas      flake8-comma      pypi          0.083333   
48      flask-openapi3     flask-openap

In [ ]:
error_rows = evidence_df[evidence_df["github_hit_count"].isna()]
print(error_rows["example_repo"].value_counts())

example_repo
ERROR 403    71
Name: count, dtype: int64


In [ ]:
import time

def search_github(candidate, filename, max_retries=3):
    query = f'"{candidate}" in:file filename:{filename}'
    for attempt in range(max_retries):
        response = requests.get(
            "https://api.github.com/search/code",
            headers=headers,
            params={"q": query}
        )
        if response.status_code == 200:
            return response
        elif response.status_code == 403:
            wait = 30 * (attempt + 1)
            print(f"  Rate limited on '{candidate}', waiting {wait}s...")
            time.sleep(wait)
        else:
            return response
    return response

# Re-run ONLY the rows that errored before
retry_rows = combined_shortlist[combined_shortlist["candidate_name"].isin(error_rows["candidate_name"]) == False]

In [ ]:
failed_candidates = evidence_df[evidence_df["github_hit_count"].isna()]["candidate_name"].tolist()
retry_shortlist = combined_shortlist[combined_shortlist["candidate_name"].isin(failed_candidates)]
print("Retrying:", len(retry_shortlist))

Retrying: 73


In [ ]:
retry_results = []

for idx, row in retry_shortlist.iterrows():
    candidate = row["candidate_name"]
    ecosystem = row["ecosystem"]
    filename = "requirements.txt" if ecosystem == "pypi" else "package.json"

    response = search_github(candidate, filename)

    if response.status_code == 200:
        data = response.json()
        hit_count = data.get("total_count", 0)
        example_repo = data["items"][0]["repository"]["full_name"] if hit_count > 0 else None
    else:
        hit_count = None
        example_repo = f"ERROR {response.status_code}"

    retry_results.append({
        "popular_name": row["popular_name"],
        "candidate_name": candidate,
        "ecosystem": ecosystem,
        "jaccard_distance": row["jaccard_distance"],
        "github_hit_count": hit_count,
        "example_repo": example_repo
    })

    time.sleep(4)  # slower pacing this time — 15/minute, safely under the 30/minute limit

retry_df = pd.DataFrame(retry_results)
print("Retry errors remaining:", retry_df["github_hit_count"].isna().sum())

  Rate limited on 'nvidia-nvtx-cu112', waiting 30s...
  Rate limited on 'requests-htmlc', waiting 30s...
  Rate limited on 'python-pptx2', waiting 30s...
  Rate limited on '@1n/is-unicode-supported', waiting 30s...
  Rate limited on '1-passport', waiting 30s...
  Rate limited on '1117http-server', waiting 30s...
  Rate limited on '1-webpack-plugin', waiting 30s...
Retry errors remaining: 0


In [2]:
from google.colab import drive
drive.mount('/content/drive')
work_dir = "/content/drive/MyDrive/npm_lsh_work"

import pandas as pd
import requests
import time

GITHUB_TOKEN = "paste_your_actual_token_here"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

combined_shortlist = pd.read_csv(f"{work_dir}/signal3_shortlist.csv")
evidence_df = pd.read_csv(f"{work_dir}/signal3_evidence_results.csv")

failed_candidates = evidence_df[evidence_df["github_hit_count"].isna()]["candidate_name"].tolist()
retry_shortlist = combined_shortlist[combined_shortlist["candidate_name"].isin(failed_candidates)]

print("Shortlist reloaded:", len(combined_shortlist))
print("Original evidence reloaded:", len(evidence_df))
print("Rows needing retry:", len(retry_shortlist))

Mounted at /content/drive
Shortlist reloaded: 150
Original evidence reloaded: 150
Rows needing retry: 73


In [3]:
print(len(GITHUB_TOKEN))

28


In [4]:
def search_github(candidate, filename, max_retries=3):
    query = f'"{candidate}" in:file filename:{filename}'
    for attempt in range(max_retries):
        response = requests.get(
            "https://api.github.com/search/code",
            headers=headers,
            params={"q": query}
        )
        if response.status_code == 200:
            return response
        elif response.status_code == 403:
            wait = 30 * (attempt + 1)
            print(f"  Rate limited on '{candidate}', waiting {wait}s...")
            time.sleep(wait)
        else:
            return response
    return response

retry_results = []

for idx, row in retry_shortlist.iterrows():
    candidate = row["candidate_name"]
    ecosystem = row["ecosystem"]
    filename = "requirements.txt" if ecosystem == "pypi" else "package.json"

    response = search_github(candidate, filename)

    if response.status_code == 200:
        data = response.json()
        hit_count = data.get("total_count", 0)
        example_repo = data["items"][0]["repository"]["full_name"] if hit_count > 0 else None
    else:
        hit_count = None
        example_repo = f"ERROR {response.status_code}"

    retry_results.append({
        "popular_name": row["popular_name"],
        "candidate_name": candidate,
        "ecosystem": ecosystem,
        "jaccard_distance": row["jaccard_distance"],
        "github_hit_count": hit_count,
        "example_repo": example_repo
    })

    # SAVE after every single row — never lose progress again, no matter what happens
    pd.DataFrame(retry_results).to_csv(f"{work_dir}/signal3_retry_progress.csv", index=False)

    time.sleep(4)

retry_df = pd.DataFrame(retry_results)
print("Retry errors remaining:", retry_df["github_hit_count"].isna().sum())

Retry errors remaining: 73


In [5]:
progress_check = pd.read_csv(f"{work_dir}/signal3_retry_progress.csv")
print(progress_check["example_repo"].value_counts())

example_repo
ERROR 401    73
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
work_dir = "/content/drive/MyDrive/npm_lsh_work"

import pandas as pd
import requests
import time

GITHUB_TOKEN = "REDACTED"
drive.mount('/content/drive')
work_dir = "/content/drive/MyDrive/npm_lsh_work"

import pandas as pd
import requests
import time

GITHUB_TOKEN = "REDACTED"
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

# Verify BEFORE doing anything else
test_response = requests.get("https://api.github.com/rate_limit", headers=headers)
print("Token check status:", test_response.status_code)

combined_shortlist = pd.read_csv(f"{work_dir}/signal3_shortlist.csv")
evidence_df = pd.read_csv(f"{work_dir}/signal3_evidence_results.csv")

failed_candidates = evidence_df[evidence_df["github_hit_count"].isna()]["candidate_name"].tolist()
retry_shortlist = combined_shortlist[combined_shortlist["candidate_name"].isin(failed_candidates)]

print("Rows needing retry:", len(retry_shortlist))
headers = {"Authorization": f"token {GITHUB_TOKEN}"}

# Verify BEFORE doing anything else
test_response = requests.get("https://api.github.com/rate_limit", headers=headers)
print("Token check status:", test_response.status_code)

combined_shortlist = pd.read_csv(f"{work_dir}/signal3_shortlist.csv")
evidence_df = pd.read_csv(f"{work_dir}/signal3_evidence_results.csv")

failed_candidates = evidence_df[evidence_df["github_hit_count"].isna()]["candidate_name"].tolist()
retry_shortlist = combined_shortlist[combined_shortlist["candidate_name"].isin(failed_candidates)]

print("Rows needing retry:", len(retry_shortlist))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Token check status: 200
Rows needing retry: 73
Token check status: 200
Rows needing retry: 73


In [8]:
def search_github(candidate, filename, max_retries=3):
    query = f'"{candidate}" in:file filename:{filename}'
    for attempt in range(max_retries):
        response = requests.get(
            "https://api.github.com/search/code",
            headers=headers,
            params={"q": query}
        )
        if response.status_code == 200:
            return response
        elif response.status_code == 403:
            wait = 30 * (attempt + 1)
            print(f"  Rate limited on '{candidate}', waiting {wait}s...")
            time.sleep(wait)
        else:
            return response
    return response

retry_results = []

for idx, row in retry_shortlist.iterrows():
    candidate = row["candidate_name"]
    ecosystem = row["ecosystem"]
    filename = "requirements.txt" if ecosystem == "pypi" else "package.json"

    response = search_github(candidate, filename)

    if response.status_code == 200:
        data = response.json()
        hit_count = data.get("total_count", 0)
        example_repo = data["items"][0]["repository"]["full_name"] if hit_count > 0 else None
    else:
        hit_count = None
        example_repo = f"ERROR {response.status_code}"

    retry_results.append({
        "popular_name": row["popular_name"],
        "candidate_name": candidate,
        "ecosystem": ecosystem,
        "jaccard_distance": row["jaccard_distance"],
        "github_hit_count": hit_count,
        "example_repo": example_repo
    })

    pd.DataFrame(retry_results).to_csv(f"{work_dir}/signal3_retry_progress.csv", index=False)
    time.sleep(4)

retry_df = pd.DataFrame(retry_results)
print("Retry errors remaining:", retry_df["github_hit_count"].isna().sum())

  Rate limited on 'nvidia-nvtx-cu112', waiting 30s...
  Rate limited on 'requests-htmlc', waiting 30s...
  Rate limited on 'python-pptx2', waiting 30s...
  Rate limited on '@1n/is-unicode-supported', waiting 30s...
  Rate limited on '1-passport', waiting 30s...
  Rate limited on '1117http-server', waiting 30s...
  Rate limited on '1-webpack-plugin', waiting 30s...
Retry errors remaining: 0


In [9]:
evidence_df_clean = evidence_df[~evidence_df["candidate_name"].isin(failed_candidates)]
evidence_df_final = pd.concat([evidence_df_clean, retry_df], ignore_index=True)

print("Total rows:", len(evidence_df_final))
print("Remaining errors:", evidence_df_final["github_hit_count"].isna().sum())

evidence_df_final.to_csv(f"{work_dir}/signal3_evidence_results.csv", index=False)
print("Saved final version — this is now your complete, error-free Signal 3 dataset")

Total rows: 150
Remaining errors: 0
Saved final version — this is now your complete, error-free Signal 3 dataset


In [10]:
real_evidence = evidence_df_final[
    (evidence_df_final["github_hit_count"] > 0) &
    (evidence_df_final["github_hit_count"] < 5000)
].sort_values("github_hit_count")

print("Candidates with PLAUSIBLE real evidence:")
print(real_evidence[["popular_name", "candidate_name", "ecosystem", "github_hit_count", "example_repo"]].to_string())

Candidates with PLAUSIBLE real evidence:
                                    popular_name                                  candidate_name ecosystem  github_hit_count                                    example_repo
34                                  pyinstrument                                   pyinstruments      pypi               1.0                                  ducminhgd/ipdb
23                   djangorestframework-jsonapi                  ak-djangorestframework-jsonapi      pypi               1.0                             tejeshn48/Naukriapp
97                                 requests-html                                  requests-htmlc      pypi               1.0                           Sergeileduc/barmanbot
96                 cuequivariance-ops-torch-cu12                   cuequivariance-ops-torch-cu13      pypi               1.0        NVIDIA-BioNeMo/BioNeMo-Inference-Runtime
83                          django-cache-memoize                           django-cache-memoiz